In [ ]:
# Figure 5: (a), (b), (c), (d), (e)

In [2]:

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ============================================================
# GLOBAL FONT SETTINGS
# ============================================================
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "stix"
plt.rcParams["axes.unicode_minus"] = False

# ============================================================
# USER SETTINGS
# ============================================================
INPUT_DIR = Path(r"C:\Users\PC1\Desktop\Code_Data\Data\Figure_5")

# Climate-zone mapping file
# Must contain columns: CityName, ClimateZone
CLIMATE_ZONE_CSV = Path(r"C:\Users\PC1\Desktop\Code_Data\Data\Figure_5\KG_city_zones\KG_climate_city.csv")

# Output folder for figures
FIGURE_DIR = INPUT_DIR / "output"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Climate-zone order: change as needed
CLIMATE_ZONE_ORDER = ["Am", "Aw", "BSh", "Cwa", "Csa"]

# Seasons to plot: choose any subset/order
ACTIVE_SEASONS = ["MAM", "JJAS", "ON"]

ACTIVE_STATISTICS = ["P90"]

# ============================================================
# COLOR SETTINGS
# ============================================================
SEASON_COLORS = {
    "MAM":  "#1f77b4",
    "JJAS": "#ff7f0e",
    "ON":   "#2ca02c",
    "DJF":  "#d62728",
}

# ============================================================
# Legend settings
# ============================================================
# Which specific scenario file should get the legend? 
# Leave as "" or None to draw on none. Set to a keyword (e.g., "ssp585_far") to restrict it.
SCENARIO_FOR_LEGEND = "ssp585_far" 

#LEGEND_LOCATION = "upper right"
LEGEND_LOCATION = "lower right"

# LEGEND_XSHIFT = 0.01     # (-0.05,0.03) for left and up, (0.03,-0.04) for right and down    # upper left
# LEGEND_YSHIFT = 0.025
LEGEND_XSHIFT = 0.01     # (-0.05,0.03) for left and up, (0.03,-0.04) for right and down    # lower right
LEGEND_YSHIFT = -0.05
# LEGEND_XSHIFT = 0.01     # (-0.05,0.03) for left and up, (0.03,-0.04) for right and down     # upper right
# LEGEND_YSHIFT = -0.05


# Boxplot style
BOX_WIDTH = 0.50
SEASON_STEP = 0.8      
CITY_GAP = 0.55       
ZONE_GAP = 1.0        
SHOW_MEAN = True       
SHOW_FLIERS = False    

# If True, cities not found in climate-zone file are appended at the end
APPEND_UNMAPPED_CITIES_AT_END = True

# ============================================================
# FONT / LABEL SIZE SETTINGS
# ============================================================

XTICK_LABEL_SIZE = 20      
YTICK_LABEL_SIZE = 20     

XLABEL_SIZE = 20         
YLABEL_SIZE = 20           

TITLE_SIZE = 12          

LEGEND_SIZE = 24         
LEGEND_TITLE_SIZE = 24

print("Configuration loaded.")
print("INPUT_DIR:", INPUT_DIR)
print("FIGURE_DIR:", FIGURE_DIR)
print("CLIMATE_ZONE_CSV:", CLIMATE_ZONE_CSV)
print("ACTIVE_SEASONS:", ACTIVE_SEASONS)
print("CLIMATE_ZONE_ORDER:", CLIMATE_ZONE_ORDER)

if not INPUT_DIR.exists():
    raise FileNotFoundError(f"Input folder not found: {INPUT_DIR}")

if not CLIMATE_ZONE_CSV.exists():
    raise FileNotFoundError(f"Climate-zone file not found: {CLIMATE_ZONE_CSV}")

# ============================================================
# HELPERS
# ============================================================

def normalize_text(x):
    return str(x).strip().lower()

def extract_base_city_name(city_name):
    """
    Convert:
        Agra__G2 -> Agra
        Agra__UHI -> Agra
        Jaipur    -> Jaipur
    """
    city_name = str(city_name).strip()
    if "__" in city_name:
        city_name = city_name.split("__", 1)[0].strip()
    return city_name

def load_climate_zone_order():
    """
    Read climate-zone mapping and return:
      - ordered city list
      - zone -> cities mapping
      - normalized city -> canonical city name mapping
    """
    cz = pd.read_csv(CLIMATE_ZONE_CSV)

    required = {"CityName", "ClimateZone"}
    missing = required - set(cz.columns)
    if missing:
        raise ValueError(f"{CLIMATE_ZONE_CSV.name} missing columns: {sorted(missing)}")

    cz = cz.copy()
    cz["CityName"] = cz["CityName"].astype(str).str.strip()
    cz["ClimateZone"] = cz["ClimateZone"].astype(str).str.strip()

    # Map normalized city name -> canonical city name from climate CSV
    city_name_map = {}
    for c in cz["CityName"].dropna().tolist():
        city_name_map[normalize_text(c)] = c

    city_order = []
    zone_to_cities = {}

    print("\nClimate-zone order summary")
    print("-" * 70)

    for zone in CLIMATE_ZONE_ORDER:
        zone_cities = (
            cz.loc[cz["ClimateZone"] == zone, "CityName"]
            .dropna()
            .sort_values()
            .tolist()
        )
        zone_to_cities[zone] = zone_cities
        city_order.extend(zone_cities)
        print(f"{zone:10s} : {len(zone_cities):3d} cities")

    return city_order, zone_to_cities, city_name_map, cz

def detect_input_files(input_dir):
    """
    The inputs:
      *_season_stats_wide.csv
      *_season_stats_long.csv
    If none are found, fall back to all CSV files.
    """
    wide_files = sorted(input_dir.glob("*_season_stats_wide.csv"))
    long_files = sorted(input_dir.glob("*_season_stats_long.csv"))

    files = []
    seen = set()

    for fp in wide_files + long_files:
        if fp not in seen:
            files.append(fp)
            seen.add(fp)

    if not files:
        files = sorted(input_dir.glob("*.csv"))

    return files

def standardize_stats_dataframe(df, source_name=""):
    """
    Accept either:
      1) wide format:
         City / CityName, Season, SeasonYear, stat columns
      2) long format:
         City / CityName, Season, SeasonYear, Statistic, Value
    Return a wide-format dataframe with one row per City-Season-Year.
    """
    df = df.copy()

    # Normalize column names
    rename_map = {}
    if "CityName" in df.columns:
        rename_map["CityName"] = "City"
    if "City" in df.columns:
        rename_map["City"] = "City"
    if rename_map:
        df = df.rename(columns=rename_map)

    required_base = {"City", "Season", "SeasonYear"}
    if not required_base.issubset(df.columns):
        raise ValueError(
            f"{source_name} missing required columns. "
            f"Found columns: {list(df.columns)}"
        )

    df["City"] = df["City"].astype(str).str.strip()
    df["Season"] = df["Season"].astype(str).str.strip().str.upper()
    df["SeasonYear"] = pd.to_numeric(df["SeasonYear"], errors="coerce")
    df = df.dropna(subset=["SeasonYear"]).copy()
    df["SeasonYear"] = df["SeasonYear"].astype(int)

    # Long format -> pivot to wide
    if {"Statistic", "Value"}.issubset(df.columns):
        print(f"Detected LONG format for: {source_name}")
        df["Statistic"] = df["Statistic"].astype(str).str.strip()

        wide_df = (
            df.pivot_table(
                index=["City", "Season", "SeasonYear"],
                columns="Statistic",
                values="Value",
                aggfunc="first",
            )
            .reset_index()
        )

        wide_df.columns.name = None
        return wide_df

    # Wide format
    print(f"Detected WIDE format for: {source_name}")
    return df

def get_stat_columns(df):
    """
    Return stat column if present.
    """
    preferred_order = ["p90"]
    meta_cols = {"City", "Season", "SeasonYear"}
    available = [c for c in df.columns if c not in meta_cols]

    ordered = [c for c in preferred_order if c in available]
    leftovers = [c for c in available if c not in ordered]
#   return ordered + leftovers
    stats = ordered + leftovers
    
    if ACTIVE_STATISTICS is None:
        return stats
    
    return [s for s in ACTIVE_STATISTICS if s in stats]
    

def season_sort_key(season):
    season = str(season).upper().strip()
    order = {s: i for i, s in enumerate(ACTIVE_SEASONS)}
    return order.get(season, 999)

def build_city_order(df, climate_city_order, zone_to_cities, city_name_map):
    """
    Build plotting city order:
      1) cities in climate-zone order
      2) any extra cities found in the data but not in climate file
    """
    available_cities_raw = sorted(df["City"].dropna().astype(str).str.strip().unique().tolist())

    # Match data city names to canonical climate-file names when possible
    canonical_available = []
    unmatched = []

    print("\nCity-name matching against climate-zone file")
    print("-" * 70)

    for raw_city in available_cities_raw:
        key = normalize_text(raw_city)
        if key in city_name_map:
            canonical = city_name_map[key]
            canonical_available.append(canonical)
            print(f"{raw_city}  -->  {canonical}")
        else:
            canonical_available.append(raw_city)
            unmatched.append(raw_city)
            print(f"{raw_city}  -->  [not found in climate-zone file]")

    available_set = set(canonical_available)

    ordered_cities = [c for c in climate_city_order if c in available_set]

    if APPEND_UNMAPPED_CITIES_AT_END:
        extras = sorted([c for c in available_set if c not in ordered_cities], key=lambda x: normalize_text(x))
        ordered_cities.extend(extras)

    print("\nOrdered cities that will be plotted:", len(ordered_cities))
    print("Cities missing from climate-zone file:", len(unmatched))

    if unmatched:
        print("Examples of unmatched cities:", unmatched[:10])

    return ordered_cities

def get_city_zone(city, zone_to_cities, city_name_map):
    """
    Return climate zone for a city, or None if not found.
    """
    city_key = normalize_text(city)
    canonical = city_name_map.get(city_key, city)

    for zone, cities in zone_to_cities.items():
        if canonical in cities:
            return zone
    return None

def compute_layout_positions(cities, zone_to_cities, city_name_map):
    """
    Compute x positions for each city and each season box.
    Adds extra gap between climate zones.
    """
    city_centers = []
    city_positions = []
    zone_boundaries = []

    current_x = 0.0
    prev_zone = None

    for city in cities:
        zone = get_city_zone(city, zone_to_cities, city_name_map)

        # Add extra separation when the zone changes
        if prev_zone is not None and zone != prev_zone:
            current_x += ZONE_GAP
            zone_boundaries.append(current_x - ZONE_GAP / 2.0)

        season_pos = {}
        for i, season in enumerate(ACTIVE_SEASONS):
            x = current_x + i * SEASON_STEP
            season_pos[season] = x

        city_center = current_x + (len(ACTIVE_SEASONS) - 1) * SEASON_STEP / 2.0
        city_centers.append(city_center)
        city_positions.append(season_pos)

        # Move to the next city block
        current_x += len(ACTIVE_SEASONS) * SEASON_STEP + CITY_GAP
        prev_zone = zone

    return city_centers, city_positions, zone_boundaries

def plot_one_file(df, source_name, out_dir, stat_cols, cities, zone_to_cities, city_name_map):
    """
    Create one figure per statistic for the given dataframe.
    """
    if df.empty:
        print(f"Skipping empty file: {source_name}")
        return []

    df = df.copy()
    df["Season"] = df["Season"].astype(str).str.strip().str.upper()
    df = df[df["Season"].isin(ACTIVE_SEASONS)].copy()

    if df.empty:
        print(f"Skipping {source_name}: no rows for ACTIVE_SEASONS={ACTIVE_SEASONS}")
        return []

    df["City"] = df["City"].astype(str).str.strip().apply(
        lambda x: city_name_map.get(normalize_text(x), x)
    )

    ordered_cities = build_city_order(df, cities, zone_to_cities, city_name_map)

    available_set = set(df["City"].dropna().astype(str).unique().tolist())
    ordered_cities = [c for c in ordered_cities if c in available_set]

    if not ordered_cities:
        print(f"Skipping {source_name}: no cities matched between data and climate-zone file.")
        return []

    print("\nPlotting file:", source_name)
    print("Total matched cities:", len(ordered_cities))
    print("Available seasons:", sorted(df["Season"].unique().tolist(), key=season_sort_key))
    print("Statistics:", stat_cols)

    # fixed layout for all stats in this file
    city_centers, city_positions, zone_boundaries = compute_layout_positions(
        ordered_cities, zone_to_cities, city_name_map
    )

    # Collect zone break positions from city order
    zone_breaks = []
    zone_sequence = [get_city_zone(c, zone_to_cities, city_name_map) for c in ordered_cities]
    for i in range(len(zone_sequence) - 1):
        if zone_sequence[i] != zone_sequence[i + 1]:
            last_city_last_box = city_positions[i][ACTIVE_SEASONS[-1]]
            next_city_first_box = city_positions[i + 1][ACTIVE_SEASONS[0]]
            zone_breaks.append((last_city_last_box + next_city_first_box) / 2.0)

    saved = []

    for stat_col in stat_cols:
        if stat_col not in df.columns:
            print(f"WARNING: statistic column '{stat_col}' not found in {source_name}. Skipping.")
            continue

        print(f"\nCreating boxplot for statistic: {stat_col}")

        data = []
        box_positions = []
        box_colors = []

        for city_idx, city in enumerate(ordered_cities):
            for season_idx, season in enumerate(ACTIVE_SEASONS):
                vals = (
                    df.loc[
                        (df["City"] == city) &
                        (df["Season"] == season),
                        stat_col
                    ]
                    .dropna()
                    .astype(float)
                    .values
                )

                if len(vals) == 0:
                    vals = np.array([np.nan])

                data.append(vals)
                x = city_positions[city_idx][season]
                box_positions.append(x)
                box_colors.append(SEASON_COLORS.get(season, "#777777"))

        # fig_width = max(18, len(ordered_cities) * 0.62)
        # fig_height = 8.5

        fig_width = max(16, len(ordered_cities) * 0.40)
        fig_height = 5.5

        fig, ax = plt.subplots(figsize=(fig_width, fig_height))

        # fig_width = max(16, len(ordered_cities) * 0.40)
        # fig_height = 4.2

        # fig, ax = plt.subplots(figsize=(fig_width, fig_height))

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=UserWarning)

            bp = ax.boxplot(
                data,
                positions=box_positions,
                widths=BOX_WIDTH,
                patch_artist=True,
                showfliers=SHOW_FLIERS,
                showmeans=SHOW_MEAN,
                meanline=True,
                manage_ticks=False,
            #     medianprops=dict(color="black", linewidth=0.004),
            #     meanprops=dict(color="black", linewidth=0.7, linestyle="-"),
            #     whiskerprops=dict(color="black", linewidth=0.7),
            #     capprops=dict(color="black", linewidth=0.7),
            #     boxprops=dict(edgecolor="black", linewidth=0.4),
            # )
                medianprops=dict(color="black", linewidth=0.004),
                meanprops=dict(color="black", linewidth=0.7, linestyle="-"),
                whiskerprops=dict(color="black", linewidth=1.0),
                capprops=dict(color="black", linewidth=1.0),
                boxprops=dict(edgecolor="black", linewidth=0.7),
            )

        # Color the boxes season-wise
        for patch, facecolor in zip(bp["boxes"], box_colors):
            patch.set_facecolor(facecolor)
            patch.set_alpha(0.75)

        # X ticks at city centers
        ax.set_xticks(city_centers)
        ax.set_xticklabels(ordered_cities, rotation=90, fontsize=XTICK_LABEL_SIZE, fontweight='bold')    #  ha='right', when rotation 90 is not used

        # Vertical separators between climate zones
        for xb in zone_breaks:
            ax.axvline(x=xb, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)

        # Tight x-limits: remove extra whitespace before first and after last city
        min_x = min(box_positions)
        max_x = max(box_positions)
        ax.set_xlim(min_x - BOX_WIDTH, max_x + BOX_WIDTH)
        ax.margins(x=0)

        ax.grid(axis="y", linestyle="--", alpha=0.35)

        # FOR SHADED ALTERNATE CITIES
        for i, xc in enumerate(city_centers):
        
            left = city_positions[i][ACTIVE_SEASONS[0]] - BOX_WIDTH*0.9
            right = city_positions[i][ACTIVE_SEASONS[-1]] + BOX_WIDTH*0.9
        
            if i % 2 == 0:
                ax.axvspan(
                    left,
                    right,
                    color="lightgray",
                    alpha=0.10,
                    zorder=0,
                )

        # Tick sizes
        ax.tick_params(axis="x", labelsize=XTICK_LABEL_SIZE)
        ax.tick_params(axis="y", labelsize=YTICK_LABEL_SIZE)
        for lbl in ax.get_yticklabels():
            lbl.set_fontweight("bold")

        # ------------------------------------------------
        # Legend (Conditionally drawn & Stylized)
        # ------------------------------------------------
        draw_legend = False
        if SCENARIO_FOR_LEGEND and SCENARIO_FOR_LEGEND.lower() in source_name.lower():
            draw_legend = True

        if draw_legend:
            legend_handles = [
                # Mathtext ensures it renders as bold Times New Roman inline
                Patch(facecolor="none", edgecolor="none", label=r"$\mathbf{Season}$"),
                *[
                    Patch(
                        facecolor=SEASON_COLORS.get(season, "#777777"),
                        edgecolor="black",
                        label=season,
                        alpha=0.75,
                    )
                    for season in ACTIVE_SEASONS
                ]
            ]
            
            ax.legend(
                handles=legend_handles,
                frameon=False,
                ncol=len(ACTIVE_SEASONS)+1,
                loc=LEGEND_LOCATION,
#                bbox_to_anchor=(LEGEND_XSHIFT, 1.0 + LEGEND_YSHIFT), # Used configured offsets, for upper left
                bbox_to_anchor=(1.0 + LEGEND_XSHIFT, LEGEND_YSHIFT), # Used configured offsets, for lower right
#                bbox_to_anchor=(1.0 + LEGEND_XSHIFT, 1.0 - LEGEND_YSHIFT), # Used configured offsets, for upper right
                fontsize=LEGEND_SIZE,
                handlelength=1.3,
                handletextpad=0.5,
                columnspacing=0.9,
            )

        fig.tight_layout()

        out_name_base = f"{Path(source_name).stem}_{stat_col}_boxplot"
        out_png = out_dir / f"{out_name_base}.png"
        out_svg = out_dir / f"{out_name_base}.svg"

        fig.savefig(out_png, dpi=1200, bbox_inches="tight", pad_inches=0.05)
        fig.savefig(out_svg, format="svg", bbox_inches="tight", pad_inches=0.05)
        plt.close(fig)

        print("Saved PNG:", out_png)
        print("Saved SVG:", out_svg)
        saved.extend([out_png, out_svg])

    return saved

# ============================================================
# MAIN
# ============================================================

city_order_from_climate, zone_to_cities, city_name_map, climate_df = load_climate_zone_order()

input_files = detect_input_files(INPUT_DIR)

print("\nFiles found for plotting:", len(input_files))
for fp in input_files:
    print(" ", fp.name)

if not input_files:
    raise FileNotFoundError(
        f"No CSV files found in {INPUT_DIR}. "
        f"Expected *_season_stats_wide.csv or *_season_stats_long.csv."
    )

all_saved_figures = []

for fp in input_files:
    print("\n" + "=" * 110)
    print("Reading:", fp.name)
    print("=" * 110)

    df_raw = pd.read_csv(fp, low_memory=False)
    print("Raw shape:", df_raw.shape)
    print("Columns:", list(df_raw.columns))

    try:
        df = standardize_stats_dataframe(df_raw, source_name=fp.name)
    except Exception as e:
        print(f"ERROR while standardizing {fp.name}: {type(e).__name__}: {e}")
        continue

    stat_cols = get_stat_columns(df)
    if not stat_cols:
        print(f"No statistic columns found in {fp.name}. Skipping.")
        continue

    print("Standardized shape:", df.shape)
    print("Stat columns detected:", stat_cols)

    try:
        saved = plot_one_file(
            df=df,
            source_name=fp.name,
            out_dir=FIGURE_DIR,
            stat_cols=stat_cols,
            cities=city_order_from_climate,
            zone_to_cities=zone_to_cities,
            city_name_map=city_name_map,
        )
        all_saved_figures.extend(saved)
    except Exception as e:
        print(f"ERROR while plotting {fp.name}: {type(e).__name__}: {e}")
        continue

print("\n" + "=" * 110)
print("Done.")
print("Total figures saved:", len(all_saved_figures) // 2)
print("Figures folder:", FIGURE_DIR)

Configuration loaded.
INPUT_DIR: C:\Users\PC1\Desktop\Code_Data\Data\Figure_5
FIGURE_DIR: C:\Users\PC1\Desktop\Code_Data\Data\Figure_5\output
CLIMATE_ZONE_CSV: C:\Users\PC1\Desktop\Code_Data\Data\Figure_5\KG_city_zones\KG_climate_city.csv
ACTIVE_SEASONS: ['MAM', 'JJAS', 'ON']
CLIMATE_ZONE_ORDER: ['Am', 'Aw', 'BSh', 'Cwa', 'Csa']

Climate-zone order summary
----------------------------------------------------------------------
Am         :   3 cities
Aw         :  17 cities
BSh        :   7 cities
Cwa        :  11 cities
Csa        :   3 cities

Files found for plotting: 5
  seasonal_percentile_UHI_historical_IMDAA_1985-2014_season_stats_wide.csv
  seasonal_percentile_UHI_ssp245_far_2070-2099_season_stats_wide.csv
  seasonal_percentile_UHI_ssp245_near_2026-2055_season_stats_wide.csv
  seasonal_percentile_UHI_ssp585_far_2070-2099_season_stats_wide.csv
  seasonal_percentile_UHI_ssp585_near_2026-2055_season_stats_wide.csv

Reading: seasonal_percentile_UHI_historical_IMDAA_1985-2014_season_